# Speculative Decoding Experiment

**Phases 1–4**: Data preparation → Baseline → AR Grid A (0.5B) → AR Grid B (1.5B)

| Config | Draft | k | Regime |
|--------|-------|---|--------|
| Baseline × 2 | — | — | det / stoch |
| Grid A × 6 | Qwen2.5-0.5B | {4,8,16} | det / stoch |
| Grid B × 6 | Qwen2.5-1.5B | {4,8,16} | det / stoch |

**Total runs:** 2 baselines + 12 speculative = 14 configurations

In [ ]:
import sys, os

# Ensure src/ is on the path
SRC_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Core imports
import torch
import pandas as pd
from pathlib import Path

from config import (
    TARGET_MODEL_ID, DRAFT_MODELS, DATASETS, REGIMES,
    DRAFT_LENGTHS, RESULTS_DIR, STABILITY_DIR, FIGURES_DIR,
    SEED, STABILITY_SEEDS, MANIFESTS_DIR,
)
from utils import set_seed, get_env_info

# Print environment
env = get_env_info()
for k, v in env.items():
    print(f"{k}: {v}")

## Phase 1 — Data Preparation & Reproducibility Lock

Load GSM8K (300), MMLU (5×100), CNN/DailyMail (200) with `seed=42`, freeze manifests.

In [ ]:
from data_loader import load_all_datasets, freeze_manifests, save_full_data, load_from_manifests

# Check if manifests already exist (skip download if so)
manifests_exist = all(
    (MANIFESTS_DIR / f"{t}_data.json").exists()
    for t in ["gsm8k", "mmlu", "cnndm"]
)

if manifests_exist:
    print("Manifests found — loading from disk (no re-download)")
    data = load_from_manifests()
else:
    print("Downloading and sampling datasets…")
    data = load_all_datasets()
    freeze_manifests(data)
    save_full_data(data)

# Quick sanity check
for task, samples in data.items():
    print(f"  {task}: {len(samples)} samples, first id: {samples[0]['sample_id']}")

### Verify Tokenizer Compatibility

In [ ]:
from data_loader import verify_tokenizer_compatibility

compatible = verify_tokenizer_compatibility()
assert compatible, "Tokenizer mismatch — cannot proceed with speculative decoding!"

## Phase 2 — Baseline Benchmarking

Run Qwen2.5-7B-Instruct autoregressive decoding on all 1,000 samples in both regimes.

In [ ]:
from baseline import load_target_model, run_baseline

# Load model once, reuse for both regimes
target_model, target_tokenizer = load_target_model()

In [ ]:
# --- Deterministic baseline ---
print("=" * 60)
print("Baseline: DETERMINISTIC regime")
print("=" * 60)
baseline_det = run_baseline(data, "deterministic", target_model, target_tokenizer)

In [ ]:
# --- Stochastic baseline ---
print("=" * 60)
print("Baseline: STOCHASTIC regime")
print("=" * 60)
baseline_stoch = run_baseline(data, "stochastic", target_model, target_tokenizer)

In [ ]:
# --- Evaluate baseline quality ---
from evaluate import evaluate_results

print("\nBaseline quality — Deterministic:")
base_quality_det = evaluate_results(baseline_det, data)

print("\nBaseline quality — Stochastic:")
base_quality_stoch = evaluate_results(baseline_stoch, data)

In [ ]:
# --- Quick latency summary ---
from metrics import compute_latency_metrics

print("\nBaseline latency — Deterministic:")
base_lat_det = compute_latency_metrics(baseline_det)
for k, v in base_lat_det.items():
    print(f"  {k}: {v}")

print("\nBaseline latency — Stochastic:")
base_lat_stoch = compute_latency_metrics(baseline_stoch)
for k, v in base_lat_stoch.items():
    print(f"  {k}: {v}")

## Phase 3 — Speculative Grid A: Qwen2.5-0.5B Draft

Run speculative decoding with 0.5B draft across k ∈ {4, 8, 16} × {deterministic, stochastic} = 6 configs.

In [ ]:
from speculative import load_draft_model, run_speculative_grid

# Load 0.5B draft model once
draft_05_model, draft_05_tokenizer = load_draft_model("0.5B")

In [ ]:
spec_results_05 = {}

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"0.5B_k{k_val}_{regime_name}"
        print(f"\n{'=' * 60}")
        print(f"Speculative: draft=0.5B, k={k_val}, regime={regime_name}")
        print(f"{'=' * 60}")
        
        results = run_speculative_grid(
            data, "0.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_05_model, draft_05_tokenizer,
        )
        spec_results_05[config_key] = results

print(f"\n✓ Grid A complete: {len(spec_results_05)} configs")

In [ ]:
# --- Grid A: Evaluate quality and compute metrics ---
from metrics import compute_acceptance_metrics, compute_speedup, compute_quality_delta

grid_a_summary = []
for config_key, results in spec_results_05.items():
    parts = config_key.split("_")  # e.g. '0.5B_k4_deterministic'
    k_val = int(parts[1][1:])
    regime = parts[2]
    
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch
    
    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)
    
    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")
    
    grid_a_summary.append({
        "config": config_key, "draft": "0.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_a = pd.DataFrame(grid_a_summary)
df_grid_a

## Phase 4 — Speculative Grid B: Qwen2.5-1.5B Draft

Run speculative decoding with 1.5B draft across the same 6 configs, plus stability analysis.

In [ ]:
# Free 0.5B draft to save memory, then load 1.5B
del draft_05_model, draft_05_tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

draft_15_model, draft_15_tokenizer = load_draft_model("1.5B")

In [ ]:
spec_results_15 = {}

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"1.5B_k{k_val}_{regime_name}"
        print(f"\n{'=' * 60}")
        print(f"Speculative: draft=1.5B, k={k_val}, regime={regime_name}")
        print(f"{'=' * 60}")
        
        results = run_speculative_grid(
            data, "1.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_15_model, draft_15_tokenizer,
        )
        spec_results_15[config_key] = results

print(f"\n✓ Grid B complete: {len(spec_results_15)} configs")

In [ ]:
# --- Grid B: Evaluate quality and compute metrics ---
grid_b_summary = []
for config_key, results in spec_results_15.items():
    parts = config_key.split("_")
    k_val = int(parts[1][1:])
    regime = parts[2]
    
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch
    
    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)
    
    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")
    
    grid_b_summary.append({
        "config": config_key, "draft": "1.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_b = pd.DataFrame(grid_b_summary)
df_grid_b

In [ ]:
# --- Combine all 12 configs into master table ---
df_all = pd.concat([df_grid_a, df_grid_b], ignore_index=True)
print("\nFull 12-config results matrix:")
display(df_all[["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"]])

# Save master table
df_all.to_csv(RESULTS_DIR / "all_configs_summary.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_summary.csv'}")

### Phase 4b — Stability Analysis (Top 2 Configs)

Identify the top-2 configs by speedup (subject to |ΔQ| ≤ 1.0) and re-run with seeds {42, 123, 999}.

In [ ]:
from metrics import compute_speedup_stability
from speculative import run_stability_analysis

# Filter configs meeting quality constraint |ΔQ| <= 1.0 for ALL tasks
delta_cols = [c for c in df_all.columns if c.startswith("delta_")]
if delta_cols:
    df_qualified = df_all[df_all[delta_cols].abs().max(axis=1) <= 1.0].copy()
else:
    # If delta columns use dQ_ prefix
    dq_cols = [c for c in df_all.columns if c.startswith("dQ_")]
    df_qualified = df_all[df_all[dq_cols].abs().max(axis=1) <= 1.0].copy()

# Select top 2 by speedup
top2 = df_qualified.nlargest(2, "S")
print("Top 2 configs (|ΔQ| ≤ 1.0):")
display(top2[["config", "S", "alpha"]])

In [ ]:
# Run stability analysis for top 2 configs
stability_results = {}

for _, row in top2.iterrows():
    draft_label = row["draft"]
    k_val = int(row["k"])
    regime = row["regime"]
    config_key = row["config"]
    
    print(f"\n{'=' * 60}")
    print(f"Stability analysis: {config_key}")
    print(f"{'=' * 60}")
    
    # Load correct draft model
    if draft_label == "0.5B":
        dm, dt = load_draft_model("0.5B")
    else:
        dm, dt = draft_15_model, draft_15_tokenizer
    
    seed_runs = run_stability_analysis(
        data, draft_label, k_val, regime,
        target_model, target_tokenizer, dm, dt,
    )
    
    # Compute speedup per seed
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    speedups = []
    for sr in seed_runs:
        s = compute_speedup(baseline_ref, sr["results"])
        speedups.append(s)
        print(f"  seed={sr['seed']}: S={s:.4f}")
    
    sigma = compute_speedup_stability(speedups)
    print(f"  σ_S = {sigma:.4f}")
    stability_results[config_key] = {"speedups": speedups, "sigma_S": sigma}

print("\n✓ Stability analysis complete")

## Summary

Display the final results matrix with all 12 configs + stability info.

In [ ]:
# Final summary table
print("=" * 80)
print("EXPERIMENT COMPLETE — FULL RESULTS MATRIX")
print("=" * 80)

display_cols = [c for c in ["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"] if c in df_all.columns]

print("\n--- Latency & Throughput ---")
display(df_all[display_cols].sort_values("R_tok_mean", ascending=False))

quality_cols = [c for c in df_all.columns if c in ["gsm8k", "mmlu", "cnndm"]]
if quality_cols:
    print("\n--- Quality (%) ---")
    display(df_all[["config"] + quality_cols])

# Best config under quality constraint
if not df_qualified.empty:
    best = df_qualified.loc[df_qualified["S"].idxmax()]
    print(f"\n★ Best config (|ΔQ| ≤ 1.0): {best['config']}")
    print(f"  Speedup: {best['S']:.2f}x, α: {best['alpha']:.3f}")

# Save combined table
df_all.to_csv(RESULTS_DIR / "all_configs_combined.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_combined.csv'}")

# Stability
if stability_results:
    print("\n--- Stability (σ_S) ---")
    for cfg, info in stability_results.items():
        print(f"  {cfg}: σ_S = {info['sigma_S']:.4f}, seeds = {info['speedups']}")